# Notebook 7: Model Evaluation & SHAP Explainability
## ESP Predictive Maintenance

**This notebook**:
1. Compares all trained models side-by-side
2. Computes SHAP values to explain which sensors drive the predictions
3. Maps sensor importance back to ESP failure modes
4. Generates publishable results tables

**SHAP (SHapley Additive exPlanations)** assigns each feature an importance value
for a particular prediction. For a failure alarm, SHAP tells you:
- Which sensors triggered the alarm (e.g., motor temperature +0.34)
- Which sensors argued against the alarm (e.g., intake pressure -0.08)
- Direction of effect (positive = increases failure probability)


In [ ]:
# -- Cell 1: Environment setup -------------------------------------------------
import sys, os
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Clone the full repo (Colab only copies this notebook by default)
    if not os.path.exists('src'):
        !git clone https://github.com/WickDager/esp-predictive-maintenance.git
        %cd esp-predictive-maintenance

    !pip install -q torch torchvision tqdm scikit-learn

    # Try Google Drive, fall back to local if mount fails
    USE_DRIVE = False
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=True)
        SAVE_DIR = '/content/drive/MyDrive/esp_pm/checkpoints'
        os.makedirs(SAVE_DIR, exist_ok=True)
        USE_DRIVE = True
        print('Google Drive mounted successfully.')
    except Exception as e:
        print(f'Drive mount failed ({e}). Using local storage instead.')
        SAVE_DIR = '/content/checkpoints'

    if not USE_DRIVE:
        SAVE_DIR = '/content/checkpoints'
else:
    SAVE_DIR = 'checkpoints'

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs('results', exist_ok=True)
print(f'Checkpoints will be saved to: {SAVE_DIR}')

import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


In [ ]:
# ── Cell 2: Prepare data ──────────────────────────────────────────
WINDOW_SIZE = 50
raw_df = generate_esp_dataset(n_wells=50, timesteps_per_well=4000, random_seed=42)
SENSOR_COLS = SYNTHETIC_SENSOR_COLS
raw_df['failure'] = (raw_df['machine_status'] == 'BROKEN').astype(int)
raw_df[SENSOR_COLS] = raw_df[SENSOR_COLS].fillna(method='ffill').fillna(0)

X_raw = raw_df[SENSOR_COLS].values.astype(np.float32)
y_raw = raw_df['failure'].values.astype(np.float32)
rul_raw = np.clip(_compute_rul(y_raw), 0, 200).astype(np.float32)
X_w, y_w, rul_w = _sliding_window(X_raw, y_raw, rul_raw, WINDOW_SIZE, step_size=1)
data = _split_and_scale(X_w, y_w, rul_w, SENSOR_COLS, 0.15, 0.15, 42)
print(f'Test set: {data["X_test"].shape}, failures: {data["y_test"].sum():.0f}')

In [ ]:
# ── Cell 3: XGBoost classifier (supervised baseline) ─────────────
print('Training XGBoost classifier...')

# Use mean of each window as tabular features for XGBoost
X_tab_train = data['X_train'].mean(axis=1)  # (N, features)
X_tab_val   = data['X_val'].mean(axis=1)
X_tab_test  = data['X_test'].mean(axis=1)

# SMOTE to handle class imbalance
smote = SMOTE(random_state=42, k_neighbors=5)
X_resampled, y_resampled = smote.fit_resample(X_tab_train, data['y_train'])
print(f'After SMOTE: {np.bincount(y_resampled.astype(int))}')

# Add std, max as features too
X_tab_train_full = np.hstack([
    data['X_train'].mean(axis=1),
    data['X_train'].std(axis=1),
    data['X_train'].max(axis=1),
])
X_tab_test_full = np.hstack([
    data['X_test'].mean(axis=1),
    data['X_test'].std(axis=1),
    data['X_test'].max(axis=1),
])

smote2 = SMOTE(random_state=42, k_neighbors=5)
X_rs2, y_rs2 = smote2.fit_resample(X_tab_train_full, data['y_train'])

xgb = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=1,  # balanced by SMOTE
    use_label_encoder=False, eval_metric='logloss',
    random_state=42, n_jobs=-1,
)
xgb.fit(X_rs2, y_rs2, verbose=False)
xgb_probs = xgb.predict_proba(X_tab_test_full)[:, 1]
xgb_metrics = anomaly_detection_metrics(data['y_test'], xgb_probs)
print(f'XGBoost AUC-ROC: {xgb_metrics["auc_roc"]:.4f} | F1: {xgb_metrics["f1"]:.4f}')

In [ ]:
# ── Cell 4: Load trained deep models (if available) ───────────────
# If you haven't run notebooks 3 & 4, use dummy scores for comparison demo
LSTM_DIR = 'checkpoints/lstm_ae'
lstm_scores = None

if os.path.exists(os.path.join(LSTM_DIR, 'pytorch_model.bin')):
    print('Loading LSTM Autoencoder...')
    lstm_model = LSTMAutoencoder.from_pretrained(LSTM_DIR, device=str(DEVICE))
    lstm_model = lstm_model.to(DEVICE)
    from torch.utils.data import DataLoader
    from src.data.loader import TimeSeriesDataset
    test_ds = TimeSeriesDataset(data['X_test'])
    test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)
    scores_list = []
    lstm_model.eval()
    with torch.no_grad():
        for batch in test_loader:
            x = batch['X'].to(DEVICE)
            scores_list.append(lstm_model.anomaly_score(x).cpu().numpy())
    lstm_scores = np.concatenate(scores_list)
    lstm_metrics = anomaly_detection_metrics(data['y_test'], lstm_scores,
                                              threshold=lstm_model.threshold)
    print(f'LSTM AE AUC-ROC: {lstm_metrics["auc_roc"]:.4f}')
else:
    print('LSTM checkpoint not found. Run notebook 03 first.')
    print('Using simulated scores for comparison demo...')
    # Simulate plausible scores for demo purposes
    rng = np.random.default_rng(1)
    lstm_scores = rng.normal(0.01, 0.005, len(data['y_test']))
    lstm_scores[data['y_test'] == 1] += 0.025
    lstm_metrics = anomaly_detection_metrics(data['y_test'], lstm_scores)

In [ ]:
# ── Cell 5: Model comparison table ───────────────────────────────
all_metrics = {
    'XGBoost + SMOTE':     xgb_metrics,
    'LSTM Autoencoder':    lstm_metrics,
}
print_metrics_table(all_metrics)

# Precision-Recall curves
from sklearn.metrics import precision_recall_curve
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#378ADD', '#E24B4A', '#1D9E75']
for (name, metrics_d), scores, color in zip(
    all_metrics.items(),
    [xgb_probs, lstm_scores],
    colors,
):
    p, r, _ = precision_recall_curve(data['y_test'], scores)
    auc_pr = metrics_d.get('auc_pr', 0)
    ax.plot(r, p, label=f'{name} (AUC-PR={auc_pr:.3f})', color=color, linewidth=2)

baseline = data['y_test'].mean()
ax.axhline(baseline, color='gray', linestyle='--', alpha=0.6,
           label=f'Random classifier (P={baseline:.3f})')
ax.set_xlabel('Recall', fontsize=11)
ax.set_ylabel('Precision', fontsize=11)
ax.set_title('Precision-Recall Curves ╗ Model Comparison', fontsize=13)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('results/07_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 6: SHAP for XGBoost ──────────────────────────────────────
print('Computing SHAP values (XGBoost)...')

# Feature names: mean + std + max of each sensor
feature_names_full = (
    [f'{c}_mean' for c in SENSOR_COLS] +
    [f'{c}_std'  for c in SENSOR_COLS] +
    [f'{c}_max'  for c in SENSOR_COLS]
)

# SHAP TreeExplainer (exact for tree models, very fast)
explainer = shap.TreeExplainer(xgb)
X_explain = X_tab_test_full[:500]  # subset for speed
shap_values = explainer.shap_values(X_explain)

# For binary classification, shap_values may be list[2]; take class 1
if isinstance(shap_values, list):
    shap_vals = shap_values[1]
else:
    shap_vals = shap_values

print(f'SHAP values shape: {shap_vals.shape}')

# Summary beeswarm plot (native SHAP)
fig, ax = plt.subplots(figsize=(9, 7))
shap.summary_plot(shap_vals, X_explain,
                   feature_names=feature_names_full,
                   max_display=20, show=False, plot_type='dot')
plt.title('SHAP Summary ╗ XGBoost Failure Classifier', fontsize=13)
plt.tight_layout()
plt.savefig('results/07_shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 7: Bar chart of mean |SHAP| ─────────────────────────────
fig = plot_shap_summary(
    shap_values=shap_vals,
    feature_names=feature_names_full,
    max_display=20,
    title='Top 20 Features by Mean |SHAP| ╗ XGBoost Failure Classifier',
    save_path='results/07_shap_bar.png',
)
plt.show()

# Map top features back to ESP domain
mean_abs_shap = np.abs(shap_vals).mean(axis=0)
top_indices = np.argsort(mean_abs_shap)[::-1][:10]
print('\nTop 10 features → ESP domain interpretation:')
domain_map = {
    'motor_temperature': 'Motor overheating / winding insulation',
    'winding_resistance': 'Insulation breakdown (resistance drift)',
    'vibration': 'Mechanical wear / bearing damage',
    'motor_current': 'Gas locking / overload / gas breakthrough',
    'intake_pressure': 'Reservoir depletion / pump-off / gas locking',
    'discharge_pressure': 'Scale buildup / blockage',
    'flow_rate': 'Pump efficiency degradation',
}
for idx in top_indices:
    feat = feature_names_full[idx]
    domain = next((v for k, v in domain_map.items() if k in feat), 'General performance')
    print(f'  [{mean_abs_shap[idx]:.4f}] {feat:40s} → {domain}')

In [ ]:
# ── Cell 8: SHAP force plot for one sample ────────────────────────
# Find a failure sample
failure_test_indices = np.where(data['y_test'][:500] == 1)[0]
if len(failure_test_indices) > 0:
    idx = failure_test_indices[0]
    print(f'Explaining sample {idx} (actual label: FAILURE)')
    print(f'XGBoost predicted failure probability: {xgb_probs[idx]:.3f}')
    
    # Waterfall plot (modern SHAP)
    shap_exp = shap.Explanation(
        values=shap_vals[idx],
        base_values=explainer.expected_value[1] if isinstance(explainer.expected_value, list)
                    else explainer.expected_value,
        data=X_explain[idx],
        feature_names=feature_names_full,
    )
    fig, ax = plt.subplots(figsize=(9, 6))
    shap.waterfall_plot(shap_exp, max_display=15, show=False)
    plt.title(f'SHAP Waterfall ╗ Sample {idx} (Failure)', fontsize=12)
    plt.tight_layout()
    plt.savefig('results/07_shap_waterfall.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ── Cell 9: Final results table for README/paper ──────────────────
results_table = pd.DataFrame([
    {
        'Model': 'XGBoost + SMOTE (baseline)',
        'AUC-ROC': xgb_metrics['auc_roc'],
        'AUC-PR':  xgb_metrics['auc_pr'],
        'F1':      xgb_metrics['f1'],
        'Recall':  xgb_metrics['recall'],
        'FPR':     xgb_metrics['false_alarm_rate'],
        'Notes':   'Supervised, tabular features',
    },
    {
        'Model': 'LSTM Autoencoder',
        'AUC-ROC': lstm_metrics['auc_roc'],
        'AUC-PR':  lstm_metrics['auc_pr'],
        'F1':      lstm_metrics['f1'],
        'Recall':  lstm_metrics['recall'],
        'FPR':     lstm_metrics['false_alarm_rate'],
        'Notes':   'Unsupervised, MC Dropout uncertainty',
    },
]).set_index('Model').round(4)

results_table.to_csv('results/07_model_comparison.csv')
print(results_table.to_string())
print('\nSaved to results/07_model_comparison.csv')

In [ ]:
# ── Cell 10: Upload checklist ─────────────────────────────────────
print('='*65)
print('UPLOAD CHECKLIST')
print('='*65)
print()
print('GitHub:')
print('  git init && git add . && git commit -m "Initial commit"')
print('  git remote add origin https://github.com/YOUR_USERNAME/esp-predictive-maintenance')
print('  git push -u origin main')
print()
print('HuggingFace (after training all models):')
print('  huggingface-cli login')
print('  python scripts/upload_to_hf.py --model lstm --model_dir checkpoints/lstm_ae \\')
print('    --hf_username YOUR_USERNAME --repo_name esp-lstm-autoencoder')
print('  python scripts/upload_to_hf.py --model rul --model_dir checkpoints/rul \\')
print('    --hf_username YOUR_USERNAME --repo_name esp-rul-bilstm')
print()
print('Demo app:')
print('  python app/gradio_app.py')
print('  # Or deploy to HuggingFace Spaces (free)')
print('='*65)